# 004 - Dataset Improvements

This notebook is the post-build cleanup pass for the raw-first dataset pipeline. It should be run after `02_build_anime_dataset` and after the first `03_catalog_eda` audit.

The goal is not to invent new metadata from nowhere. The main dataset already prioritizes MAL for popularity/score fields and AniList for genres and tags. This notebook only applies controlled fallbacks where the EDA shows weak rows: mostly missing or very sparse genre/tag labels. AniDB is treated as a fallback source because we have a local XML cache, but its raw tag system is noisy, hierarchical, and full of maintenance/container labels. For that reason, every AniDB tag is first mapped into an AniList-compatible reference table, and unmapped tags are explicitly ignored.

## What This Step Changes

- Extracts all unique AniDB tags from the local AniDB cache.
- Saves a full AniDB-to-AniList mapping table under `data/reference/anilist/`.
- Uses mapped AniDB genres only when the dataset genre field is empty.
- Uses mapped AniDB tags when the dataset tag field is empty or very sparse.
- Uses AniDB production-origin tags as a final studio/provenance fallback when `studios` is empty.
- Deletes rows that still have both `genres` and `tags` empty after all fallbacks.
- Collapses demographics to a single label and fills conservative cases from rating/genre/tag evidence.
- Filters relation and recommendation edges so they only point to MAL ids that remain in the dataset.
- Optionally enriches character/voice-actor pairs with AniList favorite counts.
- Optionally writes a slim final dataset by dropping source comparison, flag, count, `season_from_month`, `demographics_anidb_weighted`, and `production_origin` columns after repairs are applied.

This keeps AniList as the main label system while still recovering useful metadata for anime that AniList did not cover well.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

import pandas as pd
from IPython.display import display

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATASET_CSV = ROOT / "data" / "processed" / "anime_dataset.csv"
SUMMARY_FILE = ROOT / "data" / "build" / "dataset_improvement_summary.json"
ANIDB_LABEL_MAP = ROOT / "data" / "reference" / "anilist" / "anidb_to_anilist_label_map.csv"
EDA_EMPTY_AUDIT = ROOT / "data" / "build" / "audits" / "dataset_empty_field_audit.csv"

print(f"Project root: {ROOT}")
print(f"Dataset exists: {DATASET_CSV.exists()} -> {DATASET_CSV}")



def run_streaming(command, cwd=None):
    """Run a script and print stdout/stderr line-by-line while it is still running."""
    cwd = cwd or globals().get("BASE_DIR") or globals().get("ROOT") or Path.cwd()
    command = [str(part) for part in command]
    print("Running:", " ".join(command), flush=True)
    process = subprocess.Popen(
        command,
        cwd=cwd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    for line in process.stdout:
        print(line, end="", flush=True)
    return_code = process.wait()
    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, command)
    return return_code


## Baseline Missingness Before the Fallback

Before applying the fallback, we check the fields most affected by the improvement pass. This is intentionally simple: the full missingness and source-choice audit lives in `03_catalog_eda`.

In [ ]:
df_before = pd.read_csv(DATASET_CSV)

def is_missing_text(value):
    if pd.isna(value):
        return True
    text = str(value).strip()
    return text == "" or text.lower() in {"nan", "none", "null"}

def pipe_count(value):
    if is_missing_text(value):
        return 0
    return len([part for part in str(value).split("|") if part.strip()])

baseline = pd.DataFrame([
    {"field": "genres", "missing_rows": int(df_before["genres"].apply(is_missing_text).sum()), "sparse_rows_lt_3": None},
    {"field": "tags", "missing_rows": int(df_before["tags"].apply(is_missing_text).sum()), "sparse_rows_lt_3": int(df_before["tags"].apply(lambda value: pipe_count(value) < 3).sum())},
])
display(baseline)

## Run the Improvement Script

The script is the executable equivalent of this notebook step. Keeping it as a script makes the pipeline reproducible from the command line, while the notebook explains the decision logic and displays the audit outputs.

In [ ]:
cmd = [sys.executable, str(ROOT / "src" / "04_improve_anime_dataset.py")]
run_streaming(cmd)


## Improvement Summary

The summary below is the main proof that the fallback did something measurable. It records the before/after missingness, the number of AniDB tags discovered, how many were mapped, and how many rows were filled or augmented.

In [ ]:
summary = json.loads(SUMMARY_FILE.read_text(encoding="utf-8"))
print(json.dumps(summary, indent=2, ensure_ascii=False))

before_after_keys = sorted(set(summary.get("before", {})) | set(summary.get("after", {})))
summary_table = pd.DataFrame([
    {
        "metric": key,
        "before": summary.get("before", {}).get(key),
        "after": summary.get("after", {}).get(key),
    }
    for key in before_after_keys
])
display(summary_table)

action_counts = {
    key: value
    for key, value in summary.items()
    if isinstance(value, (int, float)) and key not in {"anidb_tags_total", "anidb_tags_mapped"}
}
display(pd.DataFrame(action_counts.items(), columns=["action", "count"]).sort_values("action"))

## Final Cleanup Checks

The script now performs three final consistency rules: no row may remain with both genre and tag metadata empty, demographics should be a single selected label, and graph edges should not point to anime outside the final dataset.

In [ ]:
df_after = pd.read_csv(DATASET_CSV)
valid_ids = set(df_after["mal_id"].dropna().astype(int))

empty_both = (
    df_after["genres"].apply(is_missing_text)
    & df_after["tags"].apply(is_missing_text)
).sum()

multi_demographics = df_after["demographics"].fillna("").astype(str).str.contains(r"\|").sum()

def invalid_edge_count(column):
    if column not in df_after.columns:
        return 0
    count = 0
    for value in df_after[column].fillna("").astype(str):
        for part in value.split("|"):
            if ":" not in part:
                continue
            target = part.split(":", 1)[0]
            if target.isdigit() and int(target) not in valid_ids:
                count += 1
    return count

cleanup_checks = pd.DataFrame([
    {"check": "rows_with_empty_genres_and_tags", "value": int(empty_both)},
    {"check": "rows_with_multiple_demographics", "value": int(multi_demographics)},
    {"check": "relations_edges_outside_dataset", "value": invalid_edge_count("relations")},
    {"check": "recommendation_edges_outside_dataset", "value": invalid_edge_count("recommendations")},
])
display(cleanup_checks)

## AniDB-to-AniList Mapping Audit

This table is the control surface for the fallback. If a tag looks suspicious, it should be fixed here rather than patched row by row in the dataset. The important point is that unmapped AniDB tags are not used automatically.

In [ ]:
label_map = pd.read_csv(ANIDB_LABEL_MAP)

display(label_map["target_kind"].value_counts(dropna=False).rename_axis("target_kind").reset_index(name="count"))
display(label_map["mapping_source"].value_counts(dropna=False).rename_axis("mapping_source").reset_index(name="count"))

mapped_preview = label_map[label_map["target_kind"].isin(["genre", "tag"])].copy()
mapped_preview = mapped_preview.sort_values(["target_kind", "target_label", "anidb_tag_name"])

preferred_columns = [
    "anidb_tag_id",
    "anidb_tag_name",
    "target_kind",
    "target_label",
    "mapping_source",
    "max_weight",
    "anime_count",
    "tag_occurrences",
    "fallback_weight",
]
display(mapped_preview[[column for column in preferred_columns if column in mapped_preview.columns]].head(60))

## Remaining Unmapped High-Frequency AniDB Tags

These are candidates for future manual review. Many are intentionally ignored because they are containers, maintenance tags, overly vague tags, or tags that do not match the AniList taxonomy cleanly.

In [ ]:
unmapped_review = label_map[label_map["target_kind"].eq("ignore")].copy()
sort_columns = [column for column in ["anime_count", "max_weight"] if column in unmapped_review.columns]
if sort_columns:
    unmapped_review = unmapped_review.sort_values(sort_columns, ascending=False)

preferred_columns = [
    "anidb_tag_id",
    "anidb_tag_name",
    "anime_count",
    "max_weight",
    "tag_occurrences",
    "mapping_source",
    "normalized_name",
]
display(unmapped_review[[column for column in preferred_columns if column in unmapped_review.columns]].head(80))

## Optional Voice Actor / Character / Staff Enrichment Tables

This step is intentionally separate because it may make live AniList GraphQL calls for rows whose cached media payload lacks character edges. AniList can return character favourites and voice-actor favourites inside the media `characters` connection, so this avoids the fragile Jikan `/characters/{id}` and `/people/{id}` detail-call storm.

The useful recommender object is now a separate table rather than a packed column in the anime catalog:

- `data/processed/anime_voice_actor_edges.csv`: one anime-character-voice-actor edge per row.
- `data/processed/voice_actor_index.csv`: one voice actor per row with favorite counts and voiced anime IDs.
- `data/processed/character_index.csv`: one character per row with favorite counts and associated anime IDs.
- `data/processed/anime_staff_edges.csv`: one anime-staff-role edge per row for Original Creator/Original Story, Director, and Original Character Design.
- `data/processed/staff_index.csv`: one creative staff member per row with role counts and associated anime IDs.

The builder prefers Japanese VAs, falls back to English/other languages when Japanese is missing, reuses favorite counts from the same VA/character seen in other anime, and can optionally seed favorite counts from Jikan and AniList top-list caches. Staff tables are intentionally narrower: they keep only the three creative roles we want as recommender signals, avoiding music and animator rabbit holes for now.


In [ ]:
RUN_ANILIST_CHARACTER_FAVORITE_ENRICHMENT = True
VA_CHARACTER_ROW_LIMIT = None  # set to a small number while testing; None runs the full dataset

REFRESH_JIKAN_TOP_FAVORITES = True
JIKAN_TOP_CHARACTER_PAGES = 1000  
JIKAN_TOP_PEOPLE_PAGES = 1000 

REFRESH_ANILIST_TOP_FAVORITES = True
ANILIST_TOP_PAGES = 100
ANILIST_TOP_PER_PAGE = 50

BUILD_VA_CHARACTER_TABLES = True
BUILD_STAFF_TABLES = True
MAX_VA_CHARACTER_EDGES_PER_ANIME = 30
MAX_DYNAMIC_VA_CHARACTER_EDGES_PER_ANIME = 150
DYNAMIC_VA_CHARACTER_EDGES = True

if RUN_ANILIST_CHARACTER_FAVORITE_ENRICHMENT or BUILD_VA_CHARACTER_TABLES or BUILD_STAFF_TABLES:
    cmd = [
        sys.executable,
        str(ROOT / "src" / "04_improve_anime_dataset.py"),
    ]
    if RUN_ANILIST_CHARACTER_FAVORITE_ENRICHMENT:
        cmd.append("--enrich-character-favorites")
    if BUILD_VA_CHARACTER_TABLES:
        cmd.extend([
            "--build-va-character-tables",
            "--max-va-character-edges-per-anime", str(MAX_VA_CHARACTER_EDGES_PER_ANIME),
            "--max-dynamic-va-character-edges-per-anime", str(MAX_DYNAMIC_VA_CHARACTER_EDGES_PER_ANIME),
        ])
        if not DYNAMIC_VA_CHARACTER_EDGES:
            cmd.append("--disable-dynamic-va-character-edges")
    if BUILD_STAFF_TABLES:
        cmd.append("--build-staff-tables")
    if VA_CHARACTER_ROW_LIMIT is not None:
        cmd.extend(["--character-favorite-limit", str(VA_CHARACTER_ROW_LIMIT)])
    if REFRESH_JIKAN_TOP_FAVORITES:
        cmd.extend([
            "--refresh-jikan-top-favorites",
            "--jikan-top-character-pages", str(JIKAN_TOP_CHARACTER_PAGES),
            "--jikan-top-people-pages", str(JIKAN_TOP_PEOPLE_PAGES),
        ])
    if REFRESH_ANILIST_TOP_FAVORITES:
        cmd.extend([
            "--refresh-anilist-top-favorites",
            "--anilist-top-pages", str(ANILIST_TOP_PAGES),
            "--anilist-top-per-page", str(ANILIST_TOP_PER_PAGE),
        ])
    print("Running:", " ".join(cmd), flush=True)
    process = subprocess.Popen(
        cmd,
        cwd=ROOT,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        bufsize=1,
    )
    for line in process.stdout:
        print(line, end="", flush=True)
    return_code = process.wait()
    if return_code:
        raise subprocess.CalledProcessError(return_code, cmd)
else:
    print("VA/character/staff enrichment skipped.")

edge_path = ROOT / "data" / "processed" / "anime_voice_actor_edges.csv"
va_path = ROOT / "data" / "processed" / "voice_actor_index.csv"
staff_edge_path = ROOT / "data" / "processed" / "anime_staff_edges.csv"
staff_index_path = ROOT / "data" / "processed" / "staff_index.csv"
if edge_path.exists():
    edges = pd.read_csv(edge_path)
    display(edges.head(15))
if va_path.exists():
    va_index = pd.read_csv(va_path)
    display(va_index.head(15))
if staff_edge_path.exists():
    staff_edges = pd.read_csv(staff_edge_path)
    display(staff_edges.head(15))
if staff_index_path.exists():
    staff_index = pd.read_csv(staff_index_path)
    display(staff_index.head(15))

## Refresh the EDA Outputs

Because this notebook modifies the processed dataset, the EDA audit should be refreshed afterward. This keeps `dataset_empty_field_audit.csv`, source-choice tables, and plots aligned with the final improved dataset.

In [ ]:
RUN_EDA_AFTER_IMPROVEMENTS = True

if RUN_EDA_AFTER_IMPROVEMENTS:
    cmd = [sys.executable, str(ROOT / "src" / "03_run_catalog_eda.py")]
    run_streaming(cmd)

if EDA_EMPTY_AUDIT.exists():
    display(pd.read_csv(EDA_EMPTY_AUDIT).head(30))

## Optional Final Slim Export

Use this only after you are done auditing source disagreements. It drops comparison columns such as `_mal`, `_anilist`, `_anidb`, source-specific recommendation columns, recap flags, and count columns. The selected dataset columns remain intact.

In [ ]:
DROP_AUXILIARY_COLUMNS_FOR_FINAL_EXPORT = True

if DROP_AUXILIARY_COLUMNS_FOR_FINAL_EXPORT:
    cmd = [sys.executable, str(ROOT / "src" / "04_improve_anime_dataset.py"), "--drop-auxiliary-columns"]
    run_streaming(cmd)
else:
    print("Auxiliary columns kept for auditability. Set DROP_AUXILIARY_COLUMNS_FOR_FINAL_EXPORT = True for the slim final export.")


## Interpretation

After this step, the dataset should still be understood as MAL + AniList first, with AniDB only helping where the row would otherwise be weak. If a row still has no genres or tags after this pass, that is useful information: either all three sources are sparse for that anime, or the available AniDB tags were too noisy to trust.

For recommendation work, this is better than forcing every title into a tag space. Sparse rows can be down-weighted, filtered, or handled through relations/recommendation edges rather than pretending they have reliable content metadata.